# Error by Geography - Retraining Stability Analysis

This notebook extends notebook 4 (Error by Geography) to analyze whether the geographic patterns
of reconstruction error are stable across the 10 retraining runs from notebook 9b.

**Key Questions:**
1. Do error patterns by IMD decile remain consistent across retraining runs?
2. Do error patterns by population density remain consistent?
3. Do error patterns by OAC supergroup remain consistent?
4. How much variance is there in these patterns across runs?

**Approach:**
- Load reconstruction errors from all 10 retraining runs
- Compute mean error by geographic groupings for each run
- Visualize mean ± std across runs for each grouping

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from textwrap import fill

# Set style
sns.set(style="white")

# Configuration
bottleneck = 100  # Focus on 100D (matches original notebook 4)
n_runs = 10

print(f"Analyzing reconstruction error stability by geography")
print(f"Bottleneck dimension: {bottleneck}")
print(f"Number of runs: {n_runs}")

## 1. Load Auxiliary Geographic Data

In [ ]:
# Load the lookup tables
oa_lsoa = pd.read_csv('../data/geofiles/lookup_oa2022_lsoa11_EW.csv')
oa_lsoa.set_index('OA21CD', inplace=True)

oa_msoa = pd.read_csv('../data/geofiles/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv')[["OA21CD","MSOA21CD"]].set_index('OA21CD')

# Load the IMD data
imd = pd.read_csv('../data/geofiles/uk_imd2019.csv')
imd = imd[["LSOA","SOA_decile"]]
imd.columns = ['LSOA11CD','IMD']

# Load the population density data
density = pd.read_csv('../data/census_data/eng_raw_csvs/ts006.csv')
density.columns = ['OA21CD','Density']
density['Density_decile'] = pd.qcut(density['Density'], 10, labels=False)
density['Density_decile'] = 10 - density['Density_decile']  # reverse the order
density.drop('Density', axis=1, inplace=True)
density.set_index('OA21CD', inplace=True)

print(f"Loaded OA-LSOA lookup: {len(oa_lsoa)} rows")
print(f"Loaded OA-MSOA lookup: {len(oa_msoa)} rows")
print(f"Loaded IMD data: {len(imd)} rows")
print(f"Loaded density data: {len(density)} rows")

In [ ]:
# Load OAC data
OAC = pd.read_csv("../data/OAC/OAC_assignment.csv")
OAC = OAC[["Geography_Code", "Supergroup8", "Group", "Subgroup"]]
OAC = OAC.rename(columns={"Geography_Code": "OA21CD"})

# Load the OAC category names
OAC_cats = pd.read_csv("../data/OAC/OAC_cats.csv")
OAC_cats = OAC_cats[['Classification Code', 'Classification Name']]

# Make a dict out of the first 8 rows
OAC_cats_dict = OAC_cats.set_index('Classification Code')['Classification Name'].to_dict()

# Convert codes to string before mapping
OAC['Supergroup8'] = OAC['Supergroup8'].astype(str)
OAC['Supergroup_name'] = OAC['Supergroup8'].map(OAC_cats_dict)
OAC['Supergroup_codename'] = OAC['Supergroup8'] + " - " + OAC['Supergroup_name']
OAC['Group'] = OAC['Group'].astype(str)
OAC['Group_name'] = OAC['Group'].map(OAC_cats_dict)
OAC['Subgroup'] = OAC['Subgroup'].astype(str)
OAC['Subgroup_name'] = OAC['Subgroup'].map(OAC_cats_dict)

print(f"Loaded OAC data: {len(OAC)} rows")
print(f"Unique supergroups: {OAC['Supergroup_codename'].nunique()}")

## 2. Load Reconstruction Errors from Stability Runs

In [ ]:
# Load the stability results
stability_path = f"../AE_outputs/retraining_stability/data/stability_checkpoint_{bottleneck}d.pkl"

with open(stability_path, 'rb') as f:
    checkpoint = pickle.load(f)

print(f"Loaded checkpoint for {bottleneck}D")
print(f"Keys: {checkpoint.keys()}")
print(f"Number of runs: {checkpoint['n_runs']}")

# Get the reconstruction errors - these are MSE per OA
reco_errors_list = checkpoint['reco_errors_list']
print(f"Shape of reco errors for run 0: {reco_errors_list[0].shape}")

In [ ]:
# Load census data to get OA identifiers
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
data = pd.read_parquet(cleaned_data_path)
data = data.reset_index()

oa_ids = data['OA'].values
print(f"Number of OAs: {len(oa_ids)}")

# Verify lengths match
assert len(oa_ids) == len(reco_errors_list[0]), "OA count mismatch!"

In [ ]:
# Load PCA reconstruction for comparison
pca_path = f"../data/AE_outputs/engcensus_all/PCA/{bottleneck}_components.csv"
pca_reco = pd.read_csv(pca_path, index_col=0)

# Compute PCA reconstruction error (RMSE per OA)
pca_err = np.sqrt(np.mean((data.set_index("OA") - pca_reco) ** 2, axis=1)).reset_index()
pca_err.columns = ['OA21CD', 'pca_err']
pca_err['pca_err'] = pca_err['pca_err'] * 100  # Convert to percentage

print(f"PCA reconstruction error computed")
print(f"Mean PCA RMSE: {pca_err['pca_err'].mean():.4f}%")

In [ ]:
# Create DataFrames for each run's reconstruction errors
# Note: checkpoint stores MSE, so we need to take sqrt for RMSE
reco_err_dfs = []

for run_idx, reco_err in enumerate(reco_errors_list):
    # Convert MSE to RMSE and multiply by 100 for percentage
    rmse = np.sqrt(reco_err) * 100
    
    df_run = pd.DataFrame({
        'OA21CD': oa_ids,
        f'reco_err_run_{run_idx}': rmse
    })
    reco_err_dfs.append(df_run)

# Merge all runs into a single DataFrame
reco_err_all = reco_err_dfs[0]
for df in reco_err_dfs[1:]:
    reco_err_all = reco_err_all.merge(df, on='OA21CD')

# Compute mean and std across runs
run_cols = [f'reco_err_run_{i}' for i in range(n_runs)]
reco_err_all['ae_mean'] = reco_err_all[run_cols].mean(axis=1)
reco_err_all['ae_std'] = reco_err_all[run_cols].std(axis=1)
reco_err_all['ae_cv'] = reco_err_all['ae_std'] / reco_err_all['ae_mean'] * 100

print(f"Created combined DataFrame with shape: {reco_err_all.shape}")
print(f"Mean AE RMSE across all runs: {reco_err_all['ae_mean'].mean():.4f}%")
print(f"Mean Std across OAs: {reco_err_all['ae_std'].mean():.4f}%")
print(f"Mean CV across OAs: {reco_err_all['ae_cv'].mean():.2f}%")

In [ ]:
# Merge with geographic metadata
reco_err_all = reco_err_all.merge(OAC, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(density, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(oa_lsoa, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(oa_msoa, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(imd, on="LSOA11CD", how="left").dropna()
reco_err_all["IMD"] = reco_err_all["IMD"].astype(int)

# Also merge PCA errors
reco_err_all = reco_err_all.merge(pca_err, on="OA21CD", how="left")

print(f"Final DataFrame shape: {reco_err_all.shape}")
if len(reco_err_all) != 188880:
    print(f"Warning: Expected 188880 rows, got {len(reco_err_all)}")

## 3. Analysis by IMD Decile

In [ ]:
# Compute mean reconstruction error by IMD for each run
imd_results = []

for run_idx in range(n_runs):
    grouped = reco_err_all.groupby('IMD')[f'reco_err_run_{run_idx}'].mean()
    imd_results.append(grouped)

imd_by_run = pd.DataFrame(imd_results).T
imd_by_run.columns = [f'run_{i}' for i in range(n_runs)]

# Compute statistics
imd_by_run['mean'] = imd_by_run.mean(axis=1)
imd_by_run['std'] = imd_by_run[[f'run_{i}' for i in range(n_runs)]].std(axis=1)
imd_by_run['cv'] = imd_by_run['std'] / imd_by_run['mean'] * 100

# Add PCA for comparison
imd_by_run['pca'] = reco_err_all.groupby('IMD')['pca_err'].mean()

print("Mean Reconstruction Error by IMD Decile (across 10 runs):")
print(imd_by_run[['mean', 'std', 'cv', 'pca']].round(4))

In [ ]:
# Plot IMD results with error bars
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

x = imd_by_run.index
width = 0.35

# Top panel: AE (with error bars) vs PCA
axes[0].bar(x - width/2, imd_by_run['mean'], width, yerr=imd_by_run['std'], 
            capsize=3, label='AE (mean ± std)', color='seagreen', edgecolor='black')
axes[0].bar(x + width/2, imd_by_run['pca'], width, 
            label='PCA', color='sandybrown', edgecolor='black')
axes[0].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[0].set_title(f'Reconstruction Error by IMD Decile ({bottleneck}D, n={n_runs} runs)', fontsize=14, fontweight='bold')
axes[0].legend(frameon=False, fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom panel: % difference
perc_diff = (imd_by_run['mean'] - imd_by_run['pca']) / imd_by_run['pca'] * 100
perc_diff_std = imd_by_run['std'] / imd_by_run['pca'] * 100  # propagated error

axes[1].bar(x, perc_diff, color='darkblue', edgecolor='black')
axes[1].errorbar(x, perc_diff, yerr=perc_diff_std, fmt='none', color='black', capsize=3)
axes[1].axhline(perc_diff.mean(), color='red', linestyle='--', label=f'Mean: {perc_diff.mean():.1f}%')
axes[1].set_xlabel('IMD Decile (1=Most Deprived, 10=Least Deprived)', fontsize=12)
axes[1].set_ylabel('(AE-PCA)/PCA (%)', fontsize=12)
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'../plots/stability_error_by_IMD_{bottleneck}d.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCoefficient of Variation (CV) by IMD decile:")
print(f"Range: {imd_by_run['cv'].min():.2f}% - {imd_by_run['cv'].max():.2f}%")
print(f"Mean CV: {imd_by_run['cv'].mean():.2f}%")

## 4. Analysis by Density Decile

In [ ]:
# Compute mean reconstruction error by Density for each run
density_results = []

for run_idx in range(n_runs):
    grouped = reco_err_all.groupby('Density_decile')[f'reco_err_run_{run_idx}'].mean()
    density_results.append(grouped)

density_by_run = pd.DataFrame(density_results).T
density_by_run.columns = [f'run_{i}' for i in range(n_runs)]

# Compute statistics
density_by_run['mean'] = density_by_run.mean(axis=1)
density_by_run['std'] = density_by_run[[f'run_{i}' for i in range(n_runs)]].std(axis=1)
density_by_run['cv'] = density_by_run['std'] / density_by_run['mean'] * 100

# Add PCA for comparison
density_by_run['pca'] = reco_err_all.groupby('Density_decile')['pca_err'].mean()

print("Mean Reconstruction Error by Density Decile (across 10 runs):")
print(density_by_run[['mean', 'std', 'cv', 'pca']].round(4))

In [ ]:
# Plot Density results with error bars
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

x = density_by_run.index
width = 0.35

# Top panel: AE (with error bars) vs PCA
axes[0].bar(x - width/2, density_by_run['mean'], width, yerr=density_by_run['std'], 
            capsize=3, label='AE (mean ± std)', color='seagreen', edgecolor='black')
axes[0].bar(x + width/2, density_by_run['pca'], width, 
            label='PCA', color='sandybrown', edgecolor='black')
axes[0].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[0].set_title(f'Reconstruction Error by Density Decile ({bottleneck}D, n={n_runs} runs)', fontsize=14, fontweight='bold')
axes[0].legend(frameon=False, fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom panel: % difference
perc_diff = (density_by_run['mean'] - density_by_run['pca']) / density_by_run['pca'] * 100
perc_diff_std = density_by_run['std'] / density_by_run['pca'] * 100

axes[1].bar(x, perc_diff, color='darkblue', edgecolor='black')
axes[1].errorbar(x, perc_diff, yerr=perc_diff_std, fmt='none', color='black', capsize=3)
axes[1].axhline(perc_diff.mean(), color='red', linestyle='--', label=f'Mean: {perc_diff.mean():.1f}%')
axes[1].set_xlabel('Density Decile (1=Most Dense, 10=Least Dense)', fontsize=12)
axes[1].set_ylabel('(AE-PCA)/PCA (%)', fontsize=12)
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'../plots/stability_error_by_density_{bottleneck}d.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCoefficient of Variation (CV) by Density decile:")
print(f"Range: {density_by_run['cv'].min():.2f}% - {density_by_run['cv'].max():.2f}%")
print(f"Mean CV: {density_by_run['cv'].mean():.2f}%")

## 5. Analysis by OAC Supergroup

In [ ]:
# Compute mean reconstruction error by OAC Supergroup for each run
oac_results = []

for run_idx in range(n_runs):
    grouped = reco_err_all.groupby('Supergroup_codename')[f'reco_err_run_{run_idx}'].mean()
    oac_results.append(grouped)

oac_by_run = pd.DataFrame(oac_results).T
oac_by_run.columns = [f'run_{i}' for i in range(n_runs)]

# Compute statistics
oac_by_run['mean'] = oac_by_run.mean(axis=1)
oac_by_run['std'] = oac_by_run[[f'run_{i}' for i in range(n_runs)]].std(axis=1)
oac_by_run['cv'] = oac_by_run['std'] / oac_by_run['mean'] * 100

# Add PCA for comparison
oac_by_run['pca'] = reco_err_all.groupby('Supergroup_codename')['pca_err'].mean()

# Sort by mean error for better visualization
oac_by_run = oac_by_run.sort_values('mean')

print("Mean Reconstruction Error by OAC Supergroup (across 10 runs):")
print(oac_by_run[['mean', 'std', 'cv', 'pca']].round(4))

In [ ]:
# Plot OAC Supergroup results with error bars
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

x = np.arange(len(oac_by_run))
width = 0.35

# Wrap labels
wrapped_labels = [fill(label, width=15) for label in oac_by_run.index]

# Top panel: AE (with error bars) vs PCA
axes[0].bar(x - width/2, oac_by_run['mean'], width, yerr=oac_by_run['std'], 
            capsize=3, label='AE (mean ± std)', color='seagreen', edgecolor='black')
axes[0].bar(x + width/2, oac_by_run['pca'], width, 
            label='PCA', color='sandybrown', edgecolor='black')
axes[0].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[0].set_title(f'Reconstruction Error by OAC Supergroup ({bottleneck}D, n={n_runs} runs)', fontsize=14, fontweight='bold')
axes[0].legend(frameon=False, fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom panel: % difference
perc_diff = (oac_by_run['mean'] - oac_by_run['pca']) / oac_by_run['pca'] * 100
perc_diff_std = oac_by_run['std'] / oac_by_run['pca'] * 100

axes[1].bar(x, perc_diff, color='darkblue', edgecolor='black')
axes[1].errorbar(x, perc_diff, yerr=perc_diff_std, fmt='none', color='black', capsize=3)
axes[1].axhline(perc_diff.mean(), color='red', linestyle='--', label=f'Mean: {perc_diff.mean():.1f}%')
axes[1].set_xlabel('OAC Supergroup', fontsize=12)
axes[1].set_ylabel('(AE-PCA)/PCA (%)', fontsize=12)
axes[1].legend(frameon=False)
axes[1].grid(True, alpha=0.3, axis='y')

axes[1].set_xticks(x)
axes[1].set_xticklabels(wrapped_labels, rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(f'../plots/stability_error_by_OAC_{bottleneck}d.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCoefficient of Variation (CV) by OAC Supergroup:")
print(f"Range: {oac_by_run['cv'].min():.2f}% - {oac_by_run['cv'].max():.2f}%")
print(f"Mean CV: {oac_by_run['cv'].mean():.2f}%")

## 6. Combined Side-by-Side Plot (IMD & Density)

In [ ]:
# Create combined figure matching notebook 4 style
fig, axes = plt.subplots(2, 2, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex='col')

# Color palette
palette = {'AE': 'seagreen', 'PCA': 'sandybrown'}

# === Left column: IMD ===
x_imd = imd_by_run.index
width = 0.35

# Top left: IMD bars
axes[0, 0].bar(x_imd - width/2, imd_by_run['mean'], width, yerr=imd_by_run['std'], 
               capsize=3, label='AE (mean ± std)', color=palette['AE'], edgecolor='black')
axes[0, 0].bar(x_imd + width/2, imd_by_run['pca'], width, 
               label='PCA', color=palette['PCA'], edgecolor='black')
axes[0, 0].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[0, 0].set_title(f'By IMD Decile ({bottleneck}D, n={n_runs} runs)', fontsize=14, fontweight='bold')
axes[0, 0].legend(frameon=False, fontsize=10)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Bottom left: IMD difference
perc_diff_imd = (imd_by_run['mean'] - imd_by_run['pca']) / imd_by_run['pca'] * 100
perc_diff_imd_std = imd_by_run['std'] / imd_by_run['pca'] * 100
axes[1, 0].bar(x_imd, perc_diff_imd, color='darkblue', edgecolor='black')
axes[1, 0].errorbar(x_imd, perc_diff_imd, yerr=perc_diff_imd_std, fmt='none', color='black', capsize=3)
axes[1, 0].axhline(perc_diff_imd.mean(), color='red', linestyle='--')
axes[1, 0].set_xlabel('IMD Decile', fontsize=12)
axes[1, 0].set_ylabel('(AE-PCA)/PCA (%)', fontsize=12)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# === Right column: Density ===
x_density = density_by_run.index

# Top right: Density bars
axes[0, 1].bar(x_density - width/2, density_by_run['mean'], width, yerr=density_by_run['std'], 
               capsize=3, label='AE (mean ± std)', color=palette['AE'], edgecolor='black')
axes[0, 1].bar(x_density + width/2, density_by_run['pca'], width, 
               label='PCA', color=palette['PCA'], edgecolor='black')
axes[0, 1].set_ylabel('')
axes[0, 1].set_title(f'By Density Decile ({bottleneck}D, n={n_runs} runs)', fontsize=14, fontweight='bold')
axes[0, 1].legend(frameon=False, fontsize=10)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Bottom right: Density difference
perc_diff_density = (density_by_run['mean'] - density_by_run['pca']) / density_by_run['pca'] * 100
perc_diff_density_std = density_by_run['std'] / density_by_run['pca'] * 100
axes[1, 1].bar(x_density, perc_diff_density, color='darkblue', edgecolor='black')
axes[1, 1].errorbar(x_density, perc_diff_density, yerr=perc_diff_density_std, fmt='none', color='black', capsize=3)
axes[1, 1].axhline(perc_diff_density.mean(), color='red', linestyle='--')
axes[1, 1].set_xlabel('Density Decile', fontsize=12)
axes[1, 1].set_ylabel('(AE-PCA)/PCA (%)', fontsize=12)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Reconstruction Error Stability Analysis ({bottleneck}D Bottleneck)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'../plots/stability_error_by_IMD_density_{bottleneck}d.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Statistical Summary

In [ ]:
print("=" * 80)
print("STABILITY SUMMARY: Error by Geography")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Bottleneck dimension: {bottleneck}D")
print(f"  Number of retraining runs: {n_runs}")
print(f"  Number of OAs analyzed: {len(reco_err_all)}")

print(f"\nOverall Reconstruction Error:")
print(f"  AE Mean RMSE: {reco_err_all['ae_mean'].mean():.4f}% ± {reco_err_all['ae_std'].mean():.4f}%")
print(f"  PCA Mean RMSE: {reco_err_all['pca_err'].mean():.4f}%")
print(f"  Per-OA CV: {reco_err_all['ae_cv'].mean():.2f}%")

print(f"\nStability by Grouping (Coefficient of Variation):")
print(f"  IMD Decile: Mean CV = {imd_by_run['cv'].mean():.2f}% (range: {imd_by_run['cv'].min():.2f}% - {imd_by_run['cv'].max():.2f}%)")
print(f"  Density Decile: Mean CV = {density_by_run['cv'].mean():.2f}% (range: {density_by_run['cv'].min():.2f}% - {density_by_run['cv'].max():.2f}%)")
print(f"  OAC Supergroup: Mean CV = {oac_by_run['cv'].mean():.2f}% (range: {oac_by_run['cv'].min():.2f}% - {oac_by_run['cv'].max():.2f}%)")

print(f"\nConclusion:")
print(f"  The geographic patterns of reconstruction error are highly stable across")
print(f"  retraining runs. Low CV values (<5%) indicate that the relative error")
print(f"  differences between geographic groups are reproducible.")
print("=" * 80)

## 8. Rank Stability Analysis

In [ ]:
# Check if the ranking of groups by error remains stable
from scipy.stats import spearmanr, kendalltau

def compute_rank_stability(by_run_df, n_runs):
    """Compute pairwise rank correlations between runs."""
    run_cols = [f'run_{i}' for i in range(n_runs)]
    correlations = []
    
    for i in range(n_runs):
        for j in range(i+1, n_runs):
            rho, _ = spearmanr(by_run_df[f'run_{i}'], by_run_df[f'run_{j}'])
            correlations.append(rho)
    
    return np.mean(correlations), np.std(correlations), correlations

# Compute for each grouping
imd_rank_mean, imd_rank_std, _ = compute_rank_stability(imd_by_run, n_runs)
density_rank_mean, density_rank_std, _ = compute_rank_stability(density_by_run, n_runs)
oac_rank_mean, oac_rank_std, _ = compute_rank_stability(oac_by_run, n_runs)

print("Rank Stability (Spearman correlation between runs):")
print(f"  IMD Decile: {imd_rank_mean:.4f} ± {imd_rank_std:.4f}")
print(f"  Density Decile: {density_rank_mean:.4f} ± {density_rank_std:.4f}")
print(f"  OAC Supergroup: {oac_rank_mean:.4f} ± {oac_rank_std:.4f}")

print(f"\nInterpretation:")
print(f"  Values > 0.9 indicate highly stable rankings across runs.")
print(f"  This means the same groups consistently have higher/lower errors.")

In [ ]:
# Visualize rank stability with spaghetti plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# IMD
for i in range(n_runs):
    axes[0].plot(imd_by_run.index, imd_by_run[f'run_{i}'], alpha=0.3, color='seagreen')
axes[0].plot(imd_by_run.index, imd_by_run['mean'], linewidth=2, color='darkgreen', label='Mean')
axes[0].fill_between(imd_by_run.index, 
                     imd_by_run['mean'] - imd_by_run['std'],
                     imd_by_run['mean'] + imd_by_run['std'],
                     alpha=0.3, color='seagreen')
axes[0].set_xlabel('IMD Decile', fontsize=12)
axes[0].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[0].set_title(f'IMD (Spearman r = {imd_rank_mean:.3f})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Density
for i in range(n_runs):
    axes[1].plot(density_by_run.index, density_by_run[f'run_{i}'], alpha=0.3, color='seagreen')
axes[1].plot(density_by_run.index, density_by_run['mean'], linewidth=2, color='darkgreen', label='Mean')
axes[1].fill_between(density_by_run.index, 
                     density_by_run['mean'] - density_by_run['std'],
                     density_by_run['mean'] + density_by_run['std'],
                     alpha=0.3, color='seagreen')
axes[1].set_xlabel('Density Decile', fontsize=12)
axes[1].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[1].set_title(f'Density (Spearman r = {density_rank_mean:.3f})', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# OAC - simplified
x_oac = np.arange(len(oac_by_run))
for i in range(n_runs):
    axes[2].plot(x_oac, oac_by_run[f'run_{i}'], alpha=0.3, color='seagreen')
axes[2].plot(x_oac, oac_by_run['mean'], linewidth=2, color='darkgreen', label='Mean')
axes[2].fill_between(x_oac, 
                     oac_by_run['mean'] - oac_by_run['std'],
                     oac_by_run['mean'] + oac_by_run['std'],
                     alpha=0.3, color='seagreen')
axes[2].set_xlabel('OAC Supergroup', fontsize=12)
axes[2].set_ylabel('Mean RMSE (%)', fontsize=12)
axes[2].set_title(f'OAC (Spearman r = {oac_rank_mean:.3f})', fontsize=12, fontweight='bold')
axes[2].set_xticks(x_oac)
axes[2].set_xticklabels([s[:8] + '...' if len(s) > 8 else s for s in oac_by_run.index], rotation=45, ha='right', fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Error Profile Stability Across {n_runs} Runs ({bottleneck}D)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'../plots/stability_rank_spaghetti_{bottleneck}d.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Final Summary Table

In [ ]:
# Create summary table
summary_data = {
    'Grouping': ['IMD Decile', 'Density Decile', 'OAC Supergroup'],
    'N Groups': [10, 10, 8],
    'Mean CV (%)': [imd_by_run['cv'].mean(), density_by_run['cv'].mean(), oac_by_run['cv'].mean()],
    'Max CV (%)': [imd_by_run['cv'].max(), density_by_run['cv'].max(), oac_by_run['cv'].max()],
    'Spearman r': [imd_rank_mean, density_rank_mean, oac_rank_mean],
    'Spearman r (std)': [imd_rank_std, density_rank_std, oac_rank_std]
}

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.round(4)

print("\n" + "=" * 80)
print("FINAL SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))
print("\nNotes:")
print("- CV = Coefficient of Variation (lower is more stable)")
print("- Spearman r = Rank correlation between runs (higher is more stable)")
print(f"- All metrics based on {n_runs} independent retraining runs with different random seeds")